# 081 — Aceleradores, memoria y el límite real del cómputo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

- **Roofline**: `rendimiento = min(pico_FLOPS, I × ancho_de_banda)`, donde la
  intensidad aritmética `I` = FLOPs por byte leído de HBM. El **codo** de una
  H100 SXM está en 989,5 TFLOP/s ÷ 3,35 TB/s ≈ **295 FLOP/byte**.
- **El decode vive en el suelo**: por token con lote `B` se leen todos los pesos
  y se hacen 2·N·B FLOPs → `I = 2B/b`. En FP16 con lote 1 eso es 1 FLOP/byte,
  el **0,34 %** del pico de cómputo. Sólo el lote sube la intensidad.
- **Techo de generación con lote 1**: `tokens/s ≤ ancho_de_banda / tamaño_modelo`.
  Es independiente de los TFLOPS del acelerador.
- **Jerarquía de memoria** (H100): registros y SRAM (decenas de TB/s) → L2 de
  50 MB → HBM3 de 80 GB a 3,35 TB/s → NVLink 900 GB/s → PCIe 5.0 a 64 GB/s.
- **Densa vs dispersa**: los folletos citan la cifra con dispersión 2:4, que es
  ~2× la densa. Comparar una con otra invalida cualquier conclusión.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("observability", seed=81)
show(result)


## Reflexión

1. Si tu servicio mide 480 tok/s y el techo teórico con lote 1 es 728, ¿qué
   optimizaciones dejan de tener sentido y cuáles siguen valiendo la pena?
2. Un proveedor anuncia 2× los TFLOPS de la H100 con el mismo ancho de banda.
   ¿Cuánto mejora tu decode con lote 1, y por qué?
3. ¿Por qué el prefill y el decode piden aceleradores con perfiles distintos, y
   qué implicaría separarlos en dos flotas?
